# 27a · Clean-target snapshot recovery

## The engineering question
What must be restored besides vectors, and how do we prevent a stale backup from reviving access or deleted content?

**Learning outcomes:** create/download a real Qdrant snapshot; back up SQLite with its backup API; verify bundle hashes; restore into an absent collection; remap external manifests; replay post-snapshot changes; restore alias configuration; measure RTO and journal-event RPO.

The recovery objectives were fixed in `configs/objectives.json` before this run: local RTO ≤120 seconds and zero committed source events missing after replay. RTO here starts at upload into a clean target and ends after query/policy verification. It excludes disaster detection, infrastructure provisioning, model bootstrap, and downloading a remote backup. A successful tiny local drill is not a disaster-recovery SLA.

Writers are quiesced while we capture the snapshot, ingestion manifest, and journal position. Then a source is deleted and a private grant revoked. Recovery deliberately starts from the older snapshot, replays the surviving source journal, and uses the current authoritative policy store. A separately saved policy backup is inspected but never allowed to overwrite newer revocations.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("27a", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import json, time, sqlite3
import httpx
from uuid import uuid4
from fastapi.testclient import TestClient
from qdrant_client import models as qm
from vector_masterclass.config import qdrant_url, file_hash
from vector_masterclass.evidence import json_write
from vector_masterclass.schema import Document
from vector_masterclass.store import VectorIndex
from vector_masterclass.ingestion import Ingestor
from vector_masterclass.journal import SourceJournal
from vector_masterclass.policy import PolicyStore, seed_demo_policy
from vector_masterclass.service import SearchService, create_app
from vector_masterclass.operations import switch_alias

prefix = "vmc_27a_" + uuid4().hex[:8]
source, target = VectorIndex(prefix + "_source"), VectorIndex(prefix + "_restored")
source.create()
ingest = Ingestor(
    source, lesson.path / "ingestion.sqlite", strategy="document", include_sparse=False
)
journal = SourceJournal(lesson.path / "journal.sqlite")
policy = PolicyStore(lesson.path / "policy.sqlite")
docs = [
    Document(id="manual", revision="1", title="", text="Atlas warranty lasts two years."),
    Document(id="delete-me", revision="1", title="", text="Obsolete reset advice."),
    Document(
        id="private", revision="1", title="", tenant="red", text="Red recovery token is CEDAR."
    ),
]
seed_demo_policy(policy, docs)
for doc in docs:
    journal.append(doc, expected_revision=None)
journal.replay(ingest)
alias = prefix + "_read"
switch_alias(source.client, alias, source.name)
restored = None
try:
    before = source.all_points()
    before_config = source.verify_contract().config.model_dump(mode="json")
    snapshot = source.client.create_snapshot(source.name)
    response = httpx.get(
        f"{qdrant_url()}/collections/{source.name}/snapshots/{snapshot.name}", timeout=30
    )
    response.raise_for_status()
    archive = lesson.path / "collection.snapshot"
    archive.write_bytes(response.content)
    for db, name in [(ingest.db, "ingestion-backup.sqlite"), (policy.db, "policy-backup.sqlite")]:
        with sqlite3.connect(lesson.path / name) as dest:
            db.backup(dest)
    watermark, _ = journal.snapshot()
    bundle = {
        "source_collection": source.name,
        "target_collection": target.name,
        "alias": alias,
        "watermark": watermark,
        "representation_signature": source.signature,
        "collection_config": before_config,
        "hashes": {
            name: file_hash(lesson.path / name)
            for name in ["collection.snapshot", "ingestion-backup.sqlite", "policy-backup.sqlite"]
        },
    }
    json_write(lesson.path / "bundle.json", bundle)
    # Changes after the snapshot are deliberately absent from the backup.
    tombstone = docs[1].model_copy(update={"revision": "2", "active": False})
    journal.append(tombstone, expected_revision="1")
    policy.put_source(tombstone)
    policy.revoke_grant("alice", "red", "private")
    journal.replay(ingest)
    assert all(file_hash(lesson.path / name) == digest for name, digest in bundle["hashes"].items())
    assert not target.client.collection_exists(target.name)
    started = time.perf_counter()
    with archive.open("rb") as stream:
        uploaded = httpx.post(
            f"{qdrant_url()}/collections/{target.name}/snapshots/upload",
            params={"priority": "snapshot", "wait": "true"},
            files={"snapshot": ("collection.snapshot", stream, "application/octet-stream")},
            timeout=60,
        )
    uploaded.raise_for_status()
    target.verify_contract()
    recovered = target.all_points()
    snapshot_ids_match = {p.id for p in before} == {p.id for p in recovered}
    snapshot_payloads_match = {str(p.id): p.payload for p in before} == {
        str(p.id): p.payload for p in recovered
    }
    snapshot_vectors_match = all(
        np.allclose(p.vector["dense"], q.vector["dense"], atol=1e-6)
        for p, q in zip(
            sorted(before, key=lambda p: str(p.id)), sorted(recovered, key=lambda p: str(p.id))
        )
    )
    with (
        sqlite3.connect(lesson.path / "ingestion-backup.sqlite") as backup,
        sqlite3.connect(lesson.path / "restored.sqlite") as dest,
    ):
        backup.backup(dest)
    # External manifest identifiers refer to the old physical collection.
    with sqlite3.connect(lesson.path / "restored.sqlite") as db:
        for table in ["events", "applied", "point_map"]:
            db.execute(
                f"UPDATE {table} SET collection=? WHERE collection=?", (target.name, source.name)
            )
    restored = Ingestor(
        target, lesson.path / "restored.sqlite", strategy="document", include_sparse=False
    )
    with journal.db:
        journal.db.execute(
            "INSERT INTO consumers VALUES (?,?,?)",
            (target.name, restored.pipeline_signature, watermark),
        )
    journal.replay(restored)
    switch_alias(source.client, alias, target.name)
    with TestClient(create_app(SearchService(target, policy))) as api:
        response = api.post(
            "/search",
            json={"query": "warranty recovery reset", "k": 20},
            headers={"Authorization": "Bearer demo-red-token"},
        )
        assert response.status_code == 200, response.text
        returned = {h["chunk"]["document_key"] for h in response.json()["results"]}
    rto = time.perf_counter() - started
    head, current = journal.snapshot()
    missing_events = head - journal.cursor(restored)
    consistent = restored.reconcile()["consistent"]
    aliases = {a.alias_name: a.collection_name for a in source.client.get_aliases().aliases}
    restored_config = target.verify_contract().config.model_dump(mode="json")
    config_match = all(
        before_config[key] == restored_config[key]
        for key in ["params", "hnsw_config", "quantization_config", "metadata"]
    )
    objectives = json.loads((root() / "configs/objectives.json").read_text())["recovery"]
    results = lesson.table(
        pd.DataFrame(
            [
                {
                    "snapshot_points": len(before),
                    "restored_before_replay": len(recovered),
                    "final_points": len(target.all_points()),
                    "rto_seconds": rto,
                    "missing_journal_events": missing_events,
                    "snapshot_bytes": archive.stat().st_size,
                }
            ]
        )
    )
    display(results)
    fig, ax = plt.subplots()
    ax.bar(["measured local recovery", "predeclared RTO target"], [rto, objectives["rto_seconds"]])
    ax.set(ylabel="Seconds", title="Local drill scope excludes detection and provisioning")
    lesson.figure(fig)
    plt.show()
finally:
    source.client.update_collection_aliases(
        [qm.DeleteAliasOperation(delete_alias=qm.DeleteAlias(alias_name=alias))]
    )
    if restored:
        restored.close()
    ingest.close()
    journal.close()
    policy.close()
    source.drop()
    if target.client.collection_exists(target.name):
        target.drop()
    source.close()
    target.close()

## Interpretation and exercise
The snapshot restores the old deleted point physically, but the service remains closed until replay removes it and current policy checks complete. Revoked private access stays revoked even though its stored vector survives. If current policy or the retained journal is unavailable, fail closed; restoring only an older policy backup cannot establish that no later revocations occurred.

**Exercise:** remove one journal event from a COPY and explain why the sequence-gap gate must fail. Identify which files are needed if the source disk is lost. Calculate snapshot age, retained-log recovery horizon, restore space for archive plus extracted data, and the time to download a backup over a realistic link.

A collection snapshot does not restore aliases, application configuration, model artifacts, policy history, source object storage, or all external manifests. This drill uses a clean collection on the same running server; it tests collection recovery, not a new machine/VM bootstrap or independent-site durability. Full-server restart persistence is measured separately in 05b. [Qdrant snapshot recovery](https://qdrant.tech/documentation/snapshots/).

In [ ]:
manifest = lesson.finish(
    {
        "snapshot_ids_match": snapshot_ids_match,
        "snapshot_payloads_match": snapshot_payloads_match,
        "snapshot_vectors_match": snapshot_vectors_match,
        "configuration_restored": config_match,
        "manifest_reconciles": consistent,
        "alias_restored": aliases[alias] == target.name,
        "current_policy_and_deletion_enforced": returned == {"manual@1"},
        "rpo_target": missing_events == objectives["rpo_journal_events"],
        "rto_target": rto <= objectives["rto_seconds"],
    },
    observations={
        "rto_seconds": rto,
        "rpo_missing_events": missing_events,
        "snapshot_watermark": watermark,
        "final_journal_head": head,
        "backup_hashes": bundle["hashes"],
        "clean_target": "absent collection on same Qdrant server",
        "policy_recovery": "current authoritative policy retained; stale backup not activated",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.